# 04b · GEPIC and PEPIC source aggregation

Rebuild the original four-common-crop (`mai`, `ric`, `soy`, `whe`) ISIMIP2b GEPIC and PEPIC spatial evidence independently from the NetCDF source files. Expect **16 historical and 32 future files per model**. The original three-model comparison remains a diagnostic, not a substitute for LPJmL-first route selection.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'

import re
import xarray as xr
WEIGHTS=RUNS/'spam-weights'
HIST=DATA/'isimip'/'raw'/'historical'
FUT=DATA/'isimip'/'raw'/'future'
OUT=RUNS/'multimodel'
FORCINGS={'GFDL-ESM2M':'gfdl-esm2m','HadGEM2-ES':'hadgem2-es',
          'IPSL-CM5A-LR':'ipsl-cm5a-lr','MIROC5':'miroc5'}
HIST_YEARS=[2000,2005]
FUT_YEARS=[2010,2015,2020]
COMMON_CROPS={'mai','ric','soy','whe'}
MODELS=('GEPIC','PEPIC')
LPJML_RAW=RUNS/'lpjml'/'06_GLOBAL_LPJmL_pirrww_SPAM_weighted_RAW_2000_2020.csv.gz'


In [ ]:
def file_metadata(path,period,model):
    stem=path.name.lower()
    if not stem.startswith(model.lower()+'_'):raise ValueError(path)
    crop=re.search(r'pirrww-([a-z0-9]+)-firr',stem)
    if crop is None or crop.group(1) not in COMMON_CROPS:
        raise ValueError(f'Unsupported four-common-crop source: {path.name}')
    forcing=[k for k,v in FORCINGS.items() if f'_{v}_' in stem]
    if len(forcing)!=1:raise ValueError(f'Climate forcing ambiguous: {path.name}')
    pathway='HISTORICAL' if period=='historical' else (
        'RCP26' if '_rcp26_' in stem else 'RCP60' if '_rcp60_' in stem else None)
    if pathway is None:raise ValueError(f'Unrecognised pathway: {path.name}')
    return {'crop_code':crop.group(1),'climate_forcing':forcing[0],
            'climate_scenario':pathway,'impact_model':model,'period_type':period,
            'source_file':path.name}

def years_from_source(ds):
    time=ds['time']
    units=str(time.attrs.get('units',''))
    m=re.search(r'years since\s+(\d{4})',units.lower())
    values=np.asarray(time.values,dtype=float)
    if not np.isfinite(values).all() or not np.allclose(values,np.round(values),atol=1e-8):
        raise ValueError('Source uses an invalid annual time axis')
    if m:return int(m.group(1))+np.rint(values).astype(int)
    if values.min()>=1800 and values.max()<=2500:return np.rint(values).astype(int)
    raise ValueError(f'Unsupported annual ISIMIP time coordinate: {units}')

def country_weighted(da_year,weights,metadata):
    lat=xr.DataArray(weights.lpjml_lat.to_numpy(float),dims='points')
    lon=xr.DataArray(weights.lpjml_lon.to_numpy(float),dims='points')
    values=np.asarray(da_year.sel(lat=lat,lon=lon).values,dtype=float)
    if values.ndim!=1 or len(values)!=len(weights):raise ValueError('NetCDF grid alignment failed')
    w=weights.copy()
    w['pirrww_mm_yr']=values
    w['valid']=np.isfinite(values)
    w['represented_irrigated_ha_cell']=np.where(w.valid,w.corrected_irrigated_ha,0)
    w['weighted_mm_ha']=np.where(w.valid,values*w.corrected_irrigated_ha,0)
    keys=['country_iso3','year','fable_product_norm','lpjml_crop_code']
    result=w.groupby(keys,as_index=False).agg(total_irrigated_ha=('corrected_irrigated_ha','sum'),
        represented_irrigated_ha=('represented_irrigated_ha_cell','sum'),
        weighted_mm_ha_sum=('weighted_mm_ha','sum'),grid_cells=('valid','size'),
        valid_grid_cells=('valid','sum'))
    result['spatial_coverage_fraction']=np.where(result.total_irrigated_ha>0,
        result.represented_irrigated_ha/result.total_irrigated_ha,np.nan)
    result['weighted_pirrww_mm_yr']=np.where(result.represented_irrigated_ha>0,
        result.weighted_mm_ha_sum/result.represented_irrigated_ha,np.nan)
    result['represented_water_km3']=result.weighted_mm_ha_sum*10/1e9
    result['qc_status']=np.select([result.represented_irrigated_ha<=0,
       result.spatial_coverage_fraction<0.99],['NO_COVERAGE','REVIEW_COVERAGE'],default='PASS')
    for key,val in metadata.items():result[key]=val
    return result.drop(columns='weighted_mm_ha_sum')

In [ ]:
inventory={}
for model in MODELS:
    for period,folder,expected in [('historical',HIST,16),('future',FUT,32)]:
        selected=[]
        for path in folder.rglob(model.lower()+'_*.nc4'):
            crop=re.search(r'pirrww-([a-z0-9]+)-firr',path.name.lower())
            if crop and crop.group(1) in COMMON_CROPS:selected.append(path)
        selected=sorted(selected)
        if len(selected)!=expected:
            raise ValueError(f'{model} {period}: {len(selected)} source files, expected {expected}')
        inventory[(model,period)]=selected
print({f'{m}/{p}':len(files) for (m,p),files in inventory.items()})
weights={
 'historical':pd.read_parquet(WEIGHTS/'00_FABLE_SPAM_corrected_irrigated_area_05deg_2000_2005.parquet'),
 'future':pd.read_parquet(WEIGHTS/'00_FABLE_SPAM_corrected_irrigated_area_05deg_2010_2015_2020.parquet')}
parts=[]
for (model,period),files in sorted(inventory.items()):
    for file in files:
        meta=file_metadata(file,period,model)
        subset=weights[period].loc[lambda d:d.lpjml_crop_code.eq(meta['crop_code']) &
                                 d.corrected_irrigated_ha.gt(0)]
        if subset.empty:raise ValueError(f'No SPAM support: {file.name}')
        with xr.open_dataset(file,decode_times=False,mask_and_scale=True) as ds:
            fields=[key for key in ds.data_vars if 'pirrww' in key.lower()]
            if len(fields)!=1:raise ValueError(f'Expected exactly one PIRRWW variable: {file.name}')
            annual=years_from_source(ds)
            for year in HIST_YEARS if period=='historical' else FUT_YEARS:
                idx=np.flatnonzero(annual==year)
                if len(idx)!=1:raise ValueError(f'Annual source missing/duplicate: {file.name} {year}')
                one=subset.loc[subset.year.eq(year)]
                parts.append(country_weighted(ds[fields[0]].isel(time=int(idx[0])),one,meta))
new_raw=pd.concat(parts,ignore_index=True)
if not LPJML_RAW.is_file():raise FileNotFoundError('Run notebook 04 first: '+str(LPJML_RAW))
primary=pd.read_csv(LPJML_RAW)
primary=primary[primary.crop_code.isin(COMMON_CROPS)].copy()
missing=set(primary.columns)-set(new_raw.columns)
if missing:raise ValueError(f'Output schema mismatch with LPJmL: {missing}')
combined=pd.concat([primary,new_raw[primary.columns]],ignore_index=True)
idx=['impact_model','climate_forcing','crop_code','country_iso3','year','climate_scenario','fable_product_norm']
if combined.duplicated(idx).any():raise ValueError('Duplicate three-model scientific key')
if set(combined.impact_model)!={'LPJmL','GEPIC','PEPIC'}:raise ValueError('Incomplete model support')
print('Three-model forcing-level records:',len(combined))
print(combined.groupby(['impact_model','climate_scenario','year']).size().to_string())


In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
target=OUT/'10_GLOBAL_ISIMIP2b_3MODEL_4CROP_pirrww_RAW_2000_2020.csv.gz'
if target.exists():raise FileExistsError(f'Refusing to overwrite {target}')
combined.sort_values(['impact_model','climate_forcing','crop_code','country_iso3','year','climate_scenario'],kind='stable').to_csv(target,index=False)
print('Output:',target)
